In [ ]:
from pathlib import Path
from datetime import datetime
import os, json, subprocess

# ========= CONFIG =========
SUBSCRIPTION_ID = "00000000-0000-0000-0000-000000000000"
RESOURCE_GROUP = "sample-resource-group"
LOCATION = "eastus"          # keep both resources in the same region
SKU = "S0"                   # change to "S0" if F0 creation/quota fails

TRAINING_RESOURCE_NAME = "sample-resource-name"
PREDICTION_RESOURCE_NAME = "sample-resource-name"

AZ_EXE = r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"

# ========= AZ HELPERS =========
if not os.path.exists(AZ_EXE):
    raise FileNotFoundError(f"Azure CLI executable not found at: {AZ_EXE}")

def az(args, expect_json=True):
    cmd = [AZ_EXE] + args + ["-o", "json" if expect_json else "tsv"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Azure CLI command failed:\n"
            f"CMD: {' '.join(cmd)}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )
    return json.loads(result.stdout) if expect_json else result.stdout.strip()

def az_try(args, expect_json=True):
    cmd = [AZ_EXE] + args + ["-o", "json" if expect_json else "tsv"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout) if expect_json else result.stdout.strip()

print("Checking Azure CLI login...")
acct = az_try(["account", "show"])
if acct is None:
    raise RuntimeError(
        "Azure CLI is reachable, but not logged in.\n"
        "Run in terminal:\n"
        "  az login\n"
        "or:\n"
        "  az login --use-device-code"
    )

print("Logged in as:", acct.get("user", {}).get("name"))
print("Subscription:", acct.get("name"))

print("Setting Azure subscription...")
az(["account", "set", "--subscription", SUBSCRIPTION_ID], expect_json=False)

print(f"Ensuring resource group exists: {RESOURCE_GROUP}")
rg = az([
    "group", "create",
    "--name", RESOURCE_GROUP,
    "--location", LOCATION
])

print("Resource group ready:", rg["name"])

In [ ]:
# ========= ENSURE CUSTOM VISION RESOURCE =========
def ensure_custom_vision_account(name, kind, sku, location, resource_group):
    
    # 1) Check active resource
    existing = az_try([
        "cognitiveservices", "account", "show",
        "--name", name,
        "--resource-group", resource_group
    ])

    if existing is not None:
        print(f"Already exists: {name} ({kind})")
        return existing

    # 2) Check soft-deleted resource
    deleted = az_try([
        "cognitiveservices", "account", "show-deleted",
        "--name", name,
        "--resource-group", resource_group,
        "--location", location
    ])

    if deleted is not None:
        print(f"Recovering soft-deleted resource: {name} ({kind})")
        az([
            "cognitiveservices", "account", "recover",
            "--name", name,
            "--resource-group", resource_group,
            "--location", location
        ], expect_json=False)

        # Return the recovered live resource
        return az([
            "cognitiveservices", "account", "show",
            "--name", name,
            "--resource-group", resource_group
        ])

    # 3) Create if it does not exist at all
    print(f"Creating: {name} ({kind})")
    created = az([
        "cognitiveservices", "account", "create",
        "--name", name,
        "--resource-group", resource_group,
        "--kind", kind,
        "--sku", sku,
        "--location", location
    ])
    return created

# ========= CREATE / ENSURE =========
training_account = ensure_custom_vision_account(
    TRAINING_RESOURCE_NAME,
    "CustomVision.Training",
    SKU,
    LOCATION,
    RESOURCE_GROUP
)

prediction_account = ensure_custom_vision_account(
    PREDICTION_RESOURCE_NAME,
    "CustomVision.Prediction",
    SKU,
    LOCATION,
    RESOURCE_GROUP
)

print("Custom Vision resources ready.")

In [ ]:
# ========= FETCH ENDPOINT =========
training_endpoint = az([
    "cognitiveservices", "account", "show",
    "--name", TRAINING_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "properties.endpoint"
], expect_json=False)

prediction_endpoint = az([
    "cognitiveservices", "account", "show",
    "--name", PREDICTION_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "properties.endpoint"
], expect_json=False)

# ========= FETCH KEYS =========
training_keys = az([
    "cognitiveservices", "account", "keys", "list",
    "--name", TRAINING_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP
])

prediction_keys = az([
    "cognitiveservices", "account", "keys", "list",
    "--name", PREDICTION_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP
])

# ========= FETCH PREDICTION RESOURCE ID =========
prediction_resource_id = az([
    "cognitiveservices", "account", "show",
    "--name", PREDICTION_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "id"
], expect_json=False)

# Set notebook session env vars
os.environ["VISION_TRAINING_ENDPOINT"] = training_endpoint
os.environ["VISION_PREDICTION_ENDPOINT"] = prediction_endpoint
os.environ["VISION_TRAINING_KEY"] = training_keys["key1"]
os.environ["VISION_PREDICTION_KEY"] = prediction_keys["key1"]
os.environ["VISION_PREDICTION_RESOURCE_ID"] = prediction_resource_id

# ========= SAVE TO .env FILE =========
env_path = Path(".env")

env_content = f"""VISION_TRAINING_ENDPOINT={training_endpoint}
VISION_PREDICTION_ENDPOINT=https://REDACTED_ENDPOINT/
VISION_TRAINING_KEY=REDACTED_AZURE_KEY
VISION_PREDICTION_KEY=REDACTED_AZURE_KEY
VISION_PREDICTION_RESOURCE_ID=/subscriptions/REDACTED/resourceGroups/REDACTED/providers/Microsoft.CognitiveServices/accounts/REDACTED
"""

env_path.write_text(env_content, encoding="utf-8")

print(f".env file written to: {env_path.resolve()}")
print("Environment variables set for this notebook session.\n")

print("VISION_TRAINING_ENDPOINT      =", os.environ["VISION_TRAINING_ENDPOINT"])
print("VISION_PREDICTION_ENDPOINT    =", os.environ["VISION_PREDICTION_ENDPOINT"])
print("VISION_PREDICTION_RESOURCE_ID =", os.environ["VISION_PREDICTION_RESOURCE_ID"])
print("VISION_TRAINING_KEY           =", os.environ["VISION_TRAINING_KEY"][:6] + "...")
print("VISION_PREDICTION_KEY         =", os.environ["VISION_PREDICTION_KEY"][:6] + "...")